In [ ]:
import pandas as pd
import os
from google.colab import drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Testar acesso de escrita imediatamente
import os
test_path = '/content/drive/MyDrive/colab_test_write.txt'
try:
    with open(test_path, 'w') as f:
        f.write('Testar acesso de escrita.')
    print(f"Escrita bem-sucedida em {test_path}. Acesso de escrita confirmado.")
    os.remove(test_path) # Limpar arquivo de teste
except OSError as e:
    print(f"Falha ao escrever no Google Drive: {e}. Por favor, certifique-se de que todas as permissões de escrita foram concedidas durante a montagem.")
    print("Você pode precisar revogar o acesso do Colab nas configurações de segurança da sua Conta Google e montar novamente.")

Mounted at /content/drive
Escrita bem-sucedida em /content/drive/MyDrive/colab_test_write.txt. Acesso de escrita confirmado.


In [ ]:
caminho = '/content/drive/MyDrive/Dados API/ANTAQ/'

In [ ]:
base_path = caminho
anos = range(15, 26)
dfs_soja = []

# Códigos SH que representam soja (corrigido para códigos de mercadoria)
cod_soja = ["1201", "2304", "1507", "1516", "1517"]

for i in anos:
    ano = f"20{i}"
    file_path = os.path.join(caminho, ano, f"{ano}Carga.txt")

    if os.path.exists(file_path):
        print(f"Lendo: {file_path}")

        df = pd.read_csv(file_path, sep=';', encoding='latin-1', low_memory=False)

        # Ajusta a coluna CDMercadoria para sempre ter 4 dígitos
        df["CDMercadoria"] = df["CDMercadoria"].astype(str).str.zfill(4)

        # Filtra soja
        df_soja = df[df["CDMercadoria"].isin(cod_soja)]

        print(f"Soja encontrada no ano {ano}: {df_soja.shape}")

        dfs_soja.append(df_soja)

        del df  # libera RAM
    else:
        print(f"Arquivo não encontrado: {file_path}")

# Junta tudo
if dfs_soja:
    df_soja_total = pd.concat(dfs_soja, ignore_index=True)
    print("Carga total de soja unificada:", df_soja_total.shape)
else:
    print("Nenhum dado de soja encontrado!")

Lendo: /content/drive/MyDrive/Dados API/ANTAQ/2015/2015Carga.txt
Soja encontrada no ano 2015: (7813, 27)
Lendo: /content/drive/MyDrive/Dados API/ANTAQ/2016/2016Carga.txt
Soja encontrada no ano 2016: (12372, 27)
Lendo: /content/drive/MyDrive/Dados API/ANTAQ/2017/2017Carga.txt
Soja encontrada no ano 2017: (14061, 27)
Lendo: /content/drive/MyDrive/Dados API/ANTAQ/2018/2018Carga.txt
Soja encontrada no ano 2018: (16255, 27)
Lendo: /content/drive/MyDrive/Dados API/ANTAQ/2019/2019Carga.txt
Soja encontrada no ano 2019: (16113, 27)
Lendo: /content/drive/MyDrive/Dados API/ANTAQ/2020/2020Carga.txt
Soja encontrada no ano 2020: (17746, 27)
Lendo: /content/drive/MyDrive/Dados API/ANTAQ/2021/2021Carga.txt
Soja encontrada no ano 2021: (18263, 27)
Lendo: /content/drive/MyDrive/Dados API/ANTAQ/2022/2022Carga.txt
Soja encontrada no ano 2022: (17332, 27)
Lendo: /content/drive/MyDrive/Dados API/ANTAQ/2023/2023Carga.txt
Soja encontrada no ano 2023: (21635, 27)
Lendo: /content/drive/MyDrive/Dados API/ANTAQ/2

In [ ]:
dfs_atrac = []

for i in range(15, 26):
    ano = f"20{i}"
    file_path = f"/content/drive/MyDrive/Dados API/ANTAQ/{ano}/{ano}Atracacao.txt"

    if os.path.exists(file_path):
        # Load the dataframe for the current year
        df_current_year_atrac = pd.read_csv(file_path, encoding="utf-8-sig", sep=';')
        df_current_year_atrac["ANO"] = ano
        dfs_atrac.append(df_current_year_atrac)
        # No need to del df_current_year_atrac as it's a local variable and will be garbage collected
    else:
        print(f"Atracação não encontrada: {file_path}") # Use f-string for better message

df_atrac = pd.concat(dfs_atrac, ignore_index=True)
print("Atracação carregada:", df_atrac.shape)

/tmp/ipython-input-3436680791.py:9: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  df_current_year_atrac = pd.read_csv(file_path, encoding="utf-8-sig", sep=';')
/tmp/ipython-input-3436680791.py:9: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  df_current_year_atrac = pd.read_csv(file_path, encoding="utf-8-sig", sep=';')
/tmp/ipython-input-3436680791.py:9: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  df_current_year_atrac = pd.read_csv(file_path, encoding="utf-8-sig", sep=';')
/tmp/ipython-input-3436680791.py:9: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  df_current_year_atrac = pd.read_csv(file_path, encoding="utf-8-sig", sep=';')
/tmp/ipython-input-3436680791.py:9: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=Fa

Atracação carregada: (876688, 30)


In [ ]:
# Filtro da soja exportada no Longo Curso
codigos_soja = ["1201", "2304", "1507", "1516", "1517"]

# Definindo df_merge
df_merge = pd.merge(df_soja_total, df_atrac, on='IDAtracacao', how='inner')

df_soja_exp = df_merge[
    (df_merge["CDMercadoria"].isin(codigos_soja)) &
    (df_merge["Tipo Navegacao"] == "Longo Curso") &
    (df_merge["Sentido"].isin(["Embarcados", "E"]))
].copy()

print("Total de registros filtrados:", len(df_soja_exp))

Total de registros filtrados: 132741


In [ ]:
# Códigos ANTAQ para soja e derivados
codigos_soja = ["1201", "2304", "1507", "1516", "1517"]

# Converte para string e substitui vírgulas por pontos, tratando nulos, depois converte para float
df_merge["VLPesoCargaBruta"] = (
    df_merge["VLPesoCargaBruta"]
    .fillna("0")  # opcional, para evitar erros com nulos
    .astype(str)
    .str.replace(",", ".")
    .astype(float)
)

# Filtra cargas relacionadas à soja exportada (sem exclusão de terminais)
df_soja_exp = df_merge[
    (df_merge["CDMercadoria"].isin(codigos_soja)) &
    (df_merge["Sentido"].isin(["Embarcados", "E"]))
].copy()

print("Total de registros soja exportada:", len(df_soja_exp))

# Agrupa por porto ou terminal somando peso e ordena para encontrar maiores exportadores
df_top_portos = (
    df_soja_exp.groupby("Porto Atracação")["VLPesoCargaBruta"]
    .sum()
    .reset_index()
    .sort_values(by="VLPesoCargaBruta", ascending=False)
)

print("\nTop 10 portos e terminais exportadores de soja:")
print(df_top_portos.head(10))

Total de registros soja exportada: 132741

Top 10 portos e terminais exportadores de soja:
                                      Porto Atracação  VLPesoCargaBruta
35                                             Santos      2.659318e+08
28                                          Paranaguá      1.869915e+08
24                                             Itaqui      9.946510e+07
32                                         Rio Grande      5.561122e+07
37                               São Francisco do Sul      5.362978e+07
39                       Terbian - Terminal Bianchini      5.264276e+07
52                                Terminal de Tubarão      4.518466e+07
47                        Terminal Portuário Cotegipe      3.972978e+07
40                        Terminal Graneleiro Hermasa      3.344090e+07
42  Terminal Integrador Portuário Luiz Antonio Mes...      2.755816e+07


In [ ]:
import os
import pandas as pd

caminho_base = "/content/drive/MyDrive/Dados API/ANTAQ/"

dfs_paral = []

for i in range(15, 26):  # 2015 a 2025
    ano = f"20{i}"
    file_path = os.path.join(caminho_base, ano, f"{ano}TemposAtracacaoParalisacao.txt")

    if os.path.exists(file_path):
        df_ano = pd.read_csv(file_path, sep=";", encoding="utf-8-sig")
        df_ano["ANO"] = ano
        dfs_paral.append(df_ano)
    else:
        print("Arquivo de paralisação não encontrado:", file_path)

df_paralisacoes = pd.concat(dfs_paral, ignore_index=True)
print("Paralisações carregadas:", df_paralisacoes.shape)


Paralisações carregadas: (1250749, 6)


In [ ]:
import pandas as pd

# 1) Obter a lista dos 10 maiores portos/terminais exportadores de soja do df_top_portos
# df_top_portos é gerado na célula 3PTSK7E2F_a4. Certifique-se de que ela foi executada.
top10_ports_soja = df_top_portos['Porto Atracação'].head(10).tolist()

# garantir tipos compatíveis
df_atrac['IDAtracacao'] = df_atrac['IDAtracacao'].astype(str)
df_paral['IDAtracacao'] = df_paral['IDAtracacao'].astype(str)
df_soja_exp['IDAtracacao'] = df_soja_exp['IDAtracacao'].astype(str)

# 2) df_soja_exp já contém a coluna 'Porto Atracação'
# Então, filtramos df_soja_exp diretamente para obter df_soja_top10.
df_soja_top10 = df_soja_exp[
    df_soja_exp['Porto Atracação'].isin(top10_ports_soja)
].copy()

print("Atracações de soja nesses 10 portos:", df_soja_top10.shape)

# 3) paralisações só das atracações de soja dos 10 portos
df_paral_top10 = df_paral[
    df_paral['IDAtracacao'].isin(df_soja_top10['IDAtracacao'])
].copy()

# adicionar o porto também nas paralisações
# Este merge é necessário, pois df_paral_top10 (vindo de df_paral) não contém 'Porto Atracação'.
df_paral_top10 = df_paral_top10.merge(
    df_atrac[['IDAtracacao', 'Porto Atracação']].drop_duplicates(),
    on='IDAtracacao',
    how='left'
)

print("Paralisações ligadas a soja nesses 10 portos:", df_paral_top10.shape)

# 4) agregar por Porto + motivo (você enxerga os motivos aqui)
motivo_col = 'DescricaoTempoDesconto'  # ajuste se o nome for diferente

paral_motivos = (
    df_paral_top10
    .groupby(['Porto Atracação', motivo_col])
    .agg(
        Ocorrencias=('IDTemposDescontos', 'count'),
        TotalHoras=('duration_hours', 'sum')
    )
    .reset_index()
    .sort_values(['Porto Atracação', 'TotalHoras'], ascending=[True, False])
)

# ver exemplo na tela
paral_motivos.head(50)

Atracações de soja nesses 10 portos: (71838, 56)
Paralisações ligadas a soja nesses 10 portos: (227468, 8)


,Porto Atracação,DescricaoTempoDesconto,Ocorrencias,TotalHoras
4,Itaqui,Chuva e/ou outras condições climáticas desfavo...,4740,6250.050000
16,Itaqui,"Quebra de equipamento do Operador Portuário, d...",1525,1140.366667
3,Itaqui,Arqueação,892,988.366667
0,Itaqui,Aguardando carga,1588,627.700000
15,Itaqui,Problema operacional da embarcação,37,361.750000
5,Itaqui,Falta de energia elétrica,40,95.666667
13,Itaqui,Mudança de porão,583,76.000000
6,Itaqui,Greve ou falta de trabalhadores portuários avu...,17,45.450000
10,Itaqui,Manutenção preventiva,24,41.666667
8,Itaqui,Limpeza operacional,21,34.783333


In [ ]:
# definir quantos motivos manter por porto
N = 5  # por exemplo, top 5 motivos por porto

col_porto = 'Porto Atracação'  # ou 'Complexo Portuário', conforme o seu caso

paral_motivos = (
    paral_motivos
    .sort_values([col_porto, 'Ocorrencias'], ascending=[True, False])
    .groupby(col_porto, as_index=False)
    .head(N)   # mantém só as N maiores ocorrências em cada porto
    .reset_index(drop=True)
)


In [ ]:
N = 5
col_porto = 'Porto Atracação'  # ajuste se necessário

paral_motivos = (
    paral_motivos
    .sort_values([col_porto, 'TotalHoras'], ascending=[True, False])
    .groupby(col_porto, as_index=False)
    .head(N)
    .reset_index(drop=True)
)


In [ ]:
paral_motivos.to_csv('/content/paralisacoes_soja_top_portos_maiores.csv', index=False)
print("Salvo em /content/paralisacoes_soja_top_portos_maiores.csv")


Salvo em /content/paralisacoes_soja_top_portos_maiores.csv
